# Calibración: cuando el modelo dice «90%», ¿es un 90%?

**Facsímil 7 · Evaluar, calibrar e interpretar** — capítulo 5 (calibración e incertidumbre: de *scores* a
decisiones).

Un modelo puede acertar mucho y, a la vez, **mentir con sus porcentajes**. Si vas a decidir con un umbral
(«bloquea si la probabilidad de fraude supera 0,8»), más te vale que ese 0,8 sea de verdad un 80%. Un
modelo **descalibrado** toma decisiones malas con total aplomo. En este cuaderno compruebas si las
probabilidades que da un modelo significan lo que dicen, lo mides con un *reliability diagram*, con el
**ECE** y con el **Brier score**, lo corriges con *temperature scaling* (y lo comparas con *Platt
scaling*), y ves cómo la recalibración cambia decisiones reales. La calibración es lo que convierte un
*score* en una probabilidad de la que fiarte.

### Qué vas a aprender
- Que **acertar** y estar **calibrado** son cosas distintas, y a verlo en un caso concreto.
- A leer un **reliability diagram** (lo prometido frente a lo cumplido) y a resumir el desajuste en números:
  el **ECE** y el **Brier score**.
- A **recalibrar** con *temperature scaling* sobre datos apartados, y a compararlo con *Platt scaling*.
- A comprobar que recalibrar **cambia decisiones** (un umbral) sin cambiar el acierto.
- Por qué los modelos «muy seguros de sí mismos» suelen estar peor calibrados, comparando árboles y familias
  de modelos.

### Cómo se usa
Las celdas vienen **sin ejecutar**: pulsa ▶ de arriba abajo y mira cada salida. La gracia no es que «salga»,
sino entender *por qué* sale.

### Cuánto cuesta
Unos 18 minutos. CPU, sin claves.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
np.random.seed(0)

X, y = make_classification(n_samples=4000, n_informative=6, n_redundant=2, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
modelo = RandomForestClassifier(n_estimators=60, max_depth=6, random_state=0).fit(X_tr, y_tr)
p = modelo.predict_proba(X_te)[:, 1]
print("Modelo entrenado. Acierto:", round((modelo.predict(X_te) == y_te).mean(), 3))
print("Pero, ¿son fiables sus probabilidades? Esa es OTRA pregunta, y la mas importante para decidir.")


## 1. Acertar no es estar calibrado

Imagina dos médicos que aciertan el mismo número de diagnósticos. El primero, cuando dice «90% de
probabilidad», acierta 9 de cada 10 veces que lo dice. El segundo dice «90%» pero solo acierta 6 de cada
10: sus porcentajes son adornos. Ambos «aciertan» parecido, pero solo del primero puedes fiarte para
**decidir con un umbral**. Eso es la calibración: que la probabilidad declarada coincida con la frecuencia
real de aciertos. Un modelo puede tener buen acierto y estar mal calibrado, y al revés.


## 2. El reliability diagram: prometido frente a cumplido

Agrupamos las predicciones por su probabilidad (las que rondan 0,1, las de 0,2...) y, en cada grupo, miramos
qué fracción era de verdad positiva. Si el modelo está calibrado, los puntos caen sobre la diagonal: «cuando
digo 0,7, acierto el 70%». Cuanto más se alejen, peor. El **ECE** (*expected calibration error*) resume esa
distancia media en un solo número, ponderando por cuántos casos hay en cada grupo. Definimos las dos
funciones una vez y las reutilizamos en todo el cuaderno.


In [ ]:
def reliability(prob, real, bins=10):
    bordes = np.linspace(0, 1, bins+1)
    xs, ys, pesos = [], [], []
    for i in range(bins):
        m = (prob >= bordes[i]) & (prob < bordes[i+1])
        if m.sum() > 0:
            xs.append(prob[m].mean()); ys.append(real[m].mean()); pesos.append(m.sum())
    return np.array(xs), np.array(ys), np.array(pesos)

def ece(prob, real, bins=10):
    xs, ys, pesos = reliability(prob, real, bins)
    return np.sum(pesos * np.abs(xs - ys)) / pesos.sum()

xs, ys, _ = reliability(p, y_te)
print(f"ECE (error de calibracion) del modelo: {ece(p, y_te):.3f}  (0 = perfecto)")
plt.figure(figsize=(4.6, 4.4))
plt.plot([0, 1], [0, 1], "--", color="#9c9c9c", label="calibracion perfecta")
plt.plot(xs, ys, "-o", color="black", label="el modelo")
plt.xlabel("probabilidad que dice el modelo"); plt.ylabel("frecuencia real de aciertos")
plt.title("Reliability diagram"); plt.legend(fontsize=8); plt.tight_layout(); plt.show()


**Lee la curva.** Donde la línea negra queda por debajo de la diagonal gris, el modelo es **optimista**
(promete más acierto del que cumple); por encima, es **tímido** (acierta más de lo que promete). El ECE pone
número a ese desajuste. Un ECE alto significa que sus porcentajes no se pueden tomar al pie de la letra para
decidir.


## 3. Un grupo concreto: ¿el «0,9» es de verdad un 0,9?

El reliability diagram es la foto global; vale la pena bajar a un grupo concreto y palparlo. Cogemos todas
las predicciones cuya probabilidad ronda el 0,9 (entre 0,85 y 0,95) y miramos qué fracción era de verdad
positiva. Si el modelo está calibrado en esa zona, debería salir cerca de 0,9.


In [ ]:
for centro, lo, hi in [(0.9, 0.85, 0.95), (0.7, 0.65, 0.75), (0.3, 0.25, 0.35)]:
    m = (p >= lo) & (p < hi)
    if m.sum() > 0:
        print(f"Dice ~{centro:.1f}: {m.sum():>4} casos | frecuencia real de positivos: {y_te[m].mean():.2f}")
print("\nSi el numero de la derecha no se parece al de la izquierda, ese '0,9' no es un 0,9.")


## 4. El histograma de confianza: ¿dónde se concentra el modelo?

El ECE pondera por cuántos casos hay en cada grupo, así que conviene ver el reparto. Este histograma cuenta
cuántas predicciones caen en cada nivel de confianza. Muchos modelos amontonan sus predicciones en los
extremos (cerca de 0 y de 1): cuando eso pasa y además están mal calibrados, sus errores son ruidosos
porque los comete «muy seguro».


In [ ]:
conf = np.maximum(p, 1 - p)   # confianza en la clase que predice (de 0,5 a 1)
plt.figure(figsize=(6, 3))
plt.hist(conf, bins=20, color="#444444")
plt.xlabel("confianza de la prediccion"); plt.ylabel("nº de casos")
plt.title("Histograma de confianza"); plt.tight_layout(); plt.show()
acc_real = (modelo.predict(X_te) == y_te).mean()
print(f"Confianza media del modelo: {conf.mean():.3f}   |   acierto real: {acc_real:.3f}")
if conf.mean() > acc_real:
    print("Confianza media > acierto: de media, optimista (sobreconfiado).")
else:
    print("Confianza media < acierto: de media, timido (le sobra prudencia); le vendria bien MAS confianza.")


## 5. Corregirlo: temperature scaling (sobre datos apartados)

Una técnica simple y muy usada: dividir los *logits* por una **temperatura** T que se ajusta para que las
probabilidades encajen con la realidad (es la misma temperatura del cuaderno de *sampling* del facsímil 3,
usada ahora para calibrar). No cambia **qué clase predice** el modelo (ni su acierto), solo **recalibra la
confianza**. Punto importante: la T se ajusta en un trozo de datos **apartado** (calibración) y se mide en
otro (evaluación); calibrar y medir sobre los mismos datos sería hacerse trampas al solitario.


In [ ]:
idx = np.arange(len(p))
i_cal, i_ev = train_test_split(idx, test_size=0.5, random_state=0)
eps = 1e-6
logit = np.log(np.clip(p, eps, 1-eps) / (1 - np.clip(p, eps, 1-eps)))
def aplica_T(T, lg): return 1 / (1 + np.exp(-lg / T))

Ts = np.linspace(0.5, 3.0, 26)
# Ajustamos T minimizando el ECE en el trozo de calibracion.
mejor_T = Ts[int(np.argmin([ece(aplica_T(T, logit[i_cal]), y_te[i_cal]) for T in Ts]))]
p_cal_ev = aplica_T(mejor_T, logit[i_ev])
print(f"Mejor T (ajustada en calibracion): {mejor_T:.2f}   (T>1 enfria; T<1 da mas confianza)")
print(f"ECE en evaluacion ANTES:   {ece(p[i_ev], y_te[i_ev]):.3f}")
print(f"ECE en evaluacion DESPUES: {ece(p_cal_ev, y_te[i_ev]):.3f}   (mas cerca de 0 = mas fiable)")
acc_antes = ((p[i_ev] > 0.5).astype(int) == y_te[i_ev]).mean()
acc_desp = ((p_cal_ev > 0.5).astype(int) == y_te[i_ev]).mean()
print(f"Acierto antes / despues: {acc_antes:.3f} / {acc_desp:.3f} ... la clase predicha no cambia.")


## 6. La calibración, dibujada antes y después

El número (ECE) está bien, pero se entiende mejor en la curva. Dibujamos el reliability diagram del trozo de
evaluación antes y después de recalibrar: la curva corregida debería pegarse más a la diagonal.


In [ ]:
xa, ya, _ = reliability(p[i_ev], y_te[i_ev])
xc, yc, _ = reliability(p_cal_ev, y_te[i_ev])
plt.figure(figsize=(4.6, 4.4))
plt.plot([0, 1], [0, 1], "--", color="#9c9c9c", label="perfecta")
plt.plot(xa, ya, "-o", color="#b0b0b0", label="antes")
plt.plot(xc, yc, "-o", color="black", label="despues")
plt.xlabel("probabilidad que dice el modelo"); plt.ylabel("frecuencia real")
plt.title("Reliability: antes y despues de calibrar"); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()


## 7. ¿Y esto cambia alguna decisión de verdad?

La pregunta práctica: si decido con un umbral sobre la probabilidad (por ejemplo, «revisa el caso si supera
0,8»), ¿la recalibración cambia a quién reviso? Contamos cuántos casos cruzan el umbral al recalibrar.
Si fueran cero, calibrar sería un adorno; casi nunca es cero, y por eso importa.


In [ ]:
U = 0.8
decide_antes = (p[i_ev] >= U)
decide_desp = (p_cal_ev >= U)
cambian = (decide_antes != decide_desp).sum()
print(f"Con umbral {U}: el modelo SIN calibrar marca {decide_antes.sum()} casos; calibrado, {decide_desp.sum()}.")
print(f"Decisiones que cambian al recalibrar: {cambian} de {len(i_ev)} casos.")
print("Misma clase predicha a 0,5, pero al decidir con un umbral exigente, la calibracion mueve la raya.")


## 8. Otra forma de medir: el Brier score

El ECE mira la calibración por grupos. El **Brier score** es complementario: el error cuadrático medio entre
la probabilidad y el resultado real (0 o 1). Premia a la vez **acertar** y **no pasarse de confiado**; cuanto
más bajo, mejor. Lo miramos antes y después de calibrar, en el trozo de evaluación.


In [ ]:
def brier(prob, real): return np.mean((prob - real)**2)
print(f"Brier ANTES de calibrar:   {brier(p[i_ev], y_te[i_ev]):.4f}")
print(f"Brier DESPUES de calibrar: {brier(p_cal_ev, y_te[i_ev]):.4f}   (mas bajo = mejor)")
print("El Brier resume en un numero acierto + honestidad de la confianza.")


## 9. Otra técnica: Platt scaling

*Temperature scaling* tiene un único parámetro (T). **Platt scaling** ajusta una pequeña regresión logística
sobre los *logits* (dos parámetros: pendiente y desplazamiento), así que es algo más flexible. Lo ajustamos
en el mismo trozo de calibración y comparamos su ECE con el de *temperature scaling*. No siempre gana el más
flexible: con pocos datos, el simple suele cundir más.


In [ ]:
from sklearn.linear_model import LogisticRegression
platt = LogisticRegression().fit(logit[i_cal].reshape(-1, 1), y_te[i_cal])
p_platt_ev = platt.predict_proba(logit[i_ev].reshape(-1, 1))[:, 1]
print(f"ECE sin calibrar:        {ece(p[i_ev], y_te[i_ev]):.3f}")
print(f"ECE temperature scaling: {ece(p_cal_ev, y_te[i_ev]):.3f}")
print(f"ECE Platt scaling:       {ece(p_platt_ev, y_te[i_ev]):.3f}")
print("\nDos recetas distintas para el mismo fin: que la probabilidad declarada sea honesta.")


## 10. Por qué los modelos «muy seguros» se calibran peor

Una regularidad conocida: cuanto más complejo y «seguro de sí mismo» es un modelo, peor suele estar
calibrado, porque tiende a dar probabilidades extremas (0,99 / 0,01) más de lo que la realidad justifica. Se
ve nítido con un **árbol de decisión individual**: poco profundo, sus hojas agrupan muchos casos y dan
probabilidades suaves; muy profundo, sus hojas son «puras» (todo 0 o todo 1), así que cuando se equivoca en
datos nuevos lo hace con un 100% de (falsa) confianza. Medimos su ECE según la profundidad.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
print("profundidad |  acierto | ECE (calibracion, menor = mejor)")
for prof in [2, 4, 8, None]:
    m = DecisionTreeClassifier(max_depth=prof, random_state=0).fit(X_tr, y_tr)
    pp = m.predict_proba(X_te)[:, 1]
    etiqueta = "sin lim." if prof is None else str(prof)
    print(f"   {etiqueta:>8} |  {(m.predict(X_te)==y_te).mean():.3f}  |  {ece(pp, y_te):.3f}")
print("\nEl arbol profundo da probabilidades extremas (hojas puras): peor calibrado, mas sobreconfiado.")


## 11. Distintas familias, distinta calibración

No solo influye la profundidad: cada familia de modelos tiene su «personalidad» de calibración. Comparamos
cuatro sobre los mismos datos: regresión logística (suele salir bien calibrada de fábrica), bosque
aleatorio, árbol individual y *naive bayes* (que parte de suposiciones fuertes sobre los datos). Miramos
acierto y ECE de cada uno y verás que un acierto parecido esconde calibraciones muy distintas.


In [ ]:
from sklearn.naive_bayes import GaussianNB
familias = {
    "Regresion logistica": LogisticRegression(max_iter=1000),
    "Bosque aleatorio":    RandomForestClassifier(n_estimators=60, max_depth=6, random_state=0),
    "Arbol (prof. 8)":     DecisionTreeClassifier(max_depth=8, random_state=0),
    "Naive Bayes":         GaussianNB(),
}
print(f"{'modelo':<22}{'acierto':>9}{'ECE':>9}")
for nombre, m in familias.items():
    m.fit(X_tr, y_tr)
    pp = m.predict_proba(X_te)[:, 1]
    print(f"{nombre:<22}{(m.predict(X_te)==y_te).mean():>9.3f}{ece(pp, y_te):>9.3f}")
print("\nAcierto parecido, calibracion muy distinta: por eso el ECE se mira aparte del acierto.")


## 12. Una trampa del ECE: cuántos grupos eliges

El ECE depende de en cuántos grupos (*bins*) partas el intervalo [0, 1]. Con pocos grupos, el desajuste se
promedia y parece menor; con muchos, cada grupo tiene pocos casos y la medida se vuelve ruidosa. No hay un
número mágico (10 es habitual), pero conviene saber que la cifra se mueve con esta elección.


In [ ]:
print("nº de grupos (bins) | ECE del modelo")
for b in [5, 10, 15, 20, 30]:
    print(f"        {b:>3}          |    {ece(p, y_te, bins=b):.3f}")
print("\nMismo modelo, mismas predicciones: el ECE cambia con el numero de grupos. Compara siempre con el mismo.")


## 13. Pruébalo tú

1. **Decide con umbral 0,7 en vez de 0,8** (apartado 7): ¿cambian más o menos decisiones al recalibrar? La
   zona donde la calibración aprieta depende de dónde pongas la raya.
2. **Sube el ruido del problema:** vuelve a generar los datos con `flip_y=0.2` en `make_classification` y mira
   qué le pasa al ECE del bosque. Más ruido, más difícil calibrar.
3. **Otra T a mano:** aplica `aplica_T(2.0, logit[i_ev])` y compara su ECE con el de la T óptima. Verás que
   pasarse de «enfriar» también descalibra, por el otro lado.
4. **Naive Bayes recalibrado:** entrena un `GaussianNB`, mira su ECE y aplícale *Platt scaling*. ¿Cuánto
   mejora un modelo famoso por dar probabilidades extremas?
5. **Pocos vs muchos bins** (apartado 12): elige un modelo mal calibrado y mira si el orden entre modelos se
   mantiene al cambiar el número de grupos.


## 14. Errores comunes

- **Confundir acierto y calibración.** Un modelo puede acertar mucho y mentir con sus porcentajes; son dos
  preguntas distintas (acierto, ECE).
- **Tomar las probabilidades al pie de la letra sin comprobarlas.** Si vas a decidir con un umbral, primero
  mide el ECE (o el Brier).
- **Calibrar y medir con los mismos datos.** La calibración se ajusta en un trozo apartado y se mide en otro;
  si no, te engañas.
- **Creer que recalibrar cambia las predicciones.** *Temperature scaling* no toca qué clase predice el
  modelo, solo su confianza; pero sí cambia las decisiones que dependen de un umbral.
- **Comparar ECE con distinto número de grupos.** La cifra se mueve con los *bins*: fija el mismo para todos.


## 15. Qué te llevas

- **Acierto y calibración son cosas distintas:** un modelo puede acertar mucho y aun así no ser fiable en sus
  probabilidades.
- El **reliability diagram**, el **ECE** y el **Brier score** miden si «0,8» significa de verdad 80%; el
  *temperature scaling* (o el *Platt scaling*) lo corrige sin tocar las predicciones, ajustado en datos
  apartados.
- Recalibrar **cambia decisiones reales** cuando dependen de un umbral, aunque el acierto no se mueva.
- Los modelos **más seguros de sí mismos** (como los árboles profundos) tienden a estar peor calibrados:
  dan probabilidades exageradas, y cada familia de modelos tiene su propia «personalidad» de calibración.
- Si decides con un umbral sobre la probabilidad, **necesitas** que esa probabilidad sea honesta.

**Para seguir:** el cuaderno del coste del error usa estas probabilidades para elegir el umbral que minimiza
el coste real de equivocarse; el capítulo 6 trata la interpretabilidad.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*